In [1]:
import numpy as np
import pandas as pd

# Binary Classification with a Bank Churn Dataset

train = pd.read_csv(
    r"C:\Users\IVAN\Desktop\Machine-Learning-Roadmap-Theory-and-Practice\Files\Binary Classification with a Bank Churn Dataset\train.csv",
    sep=','
)

X_test = pd.read_csv(
    r"C:\Users\IVAN\Desktop\Machine-Learning-Roadmap-Theory-and-Practice\Files\Binary Classification with a Bank Churn Dataset\test.csv",
    sep=','
)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# X_train.info()
# X_train.describe()

X_train = train.drop(columns=['Exited'])
y_train = train['Exited']

X_train = X_train.drop(columns=['id', 'CustomerId', 'Surname'])
X_test = X_test.drop(columns=['id', 'CustomerId', 'Surname'])

print(X_train['Geography'].unique())
print(X_train['Gender'].unique())
print(X_train.shape)

X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

cat_features = ['Geography', 'Gender']

# for i in cat_features:
#     print(X_tr[i].dtype)
# X_train.head()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str
<StringArray>
['Male', 'Female']
Length: 2, dtype: str
(165034, 10)


In [3]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier


model = CatBoostClassifier(
    iterations=500,
    cat_features=cat_features,
    random_state=42
)

model.fit(X_tr, y_tr)

print('Train acc:', accuracy_score(model.predict(X_tr), y_tr))
print('Val acc:', accuracy_score(model.predict(X_te), y_te))

y_pred_proba = model.predict_proba(X_te)[:, 1]
roc_auc = roc_auc_score(y_te, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

Learning rate set to 0.156495
0:	learn: 0.5665335	total: 217ms	remaining: 1m 48s
1:	learn: 0.4875305	total: 274ms	remaining: 1m 8s
2:	learn: 0.4338593	total: 332ms	remaining: 55s
3:	learn: 0.4003569	total: 387ms	remaining: 48s
4:	learn: 0.3768449	total: 447ms	remaining: 44.3s
5:	learn: 0.3616209	total: 512ms	remaining: 42.2s
6:	learn: 0.3511948	total: 577ms	remaining: 40.6s
7:	learn: 0.3440154	total: 640ms	remaining: 39.4s
8:	learn: 0.3389373	total: 701ms	remaining: 38.3s
9:	learn: 0.3356159	total: 762ms	remaining: 37.3s
10:	learn: 0.3324036	total: 824ms	remaining: 36.6s
11:	learn: 0.3298442	total: 888ms	remaining: 36.1s
12:	learn: 0.3280948	total: 953ms	remaining: 35.7s
13:	learn: 0.3269245	total: 1.01s	remaining: 35.2s
14:	learn: 0.3259745	total: 1.08s	remaining: 35s
15:	learn: 0.3251354	total: 1.14s	remaining: 34.6s
16:	learn: 0.3243785	total: 1.21s	remaining: 34.3s
17:	learn: 0.3238969	total: 1.27s	remaining: 34.1s
18:	learn: 0.3233900	total: 1.33s	remaining: 33.8s
19:	learn: 0.322

In [4]:
oof_preds = np.zeros(len(X_train))
fold_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_kf = CatBoostClassifier(
        iterations=500,
        eval_metric='AUC',
        cat_features=cat_features,
        random_state = 42,
        verbose=0
    )

    model_kf.fit(
        X_tr, y_tr,
        eval_set = (X_val, y_val),
        early_stopping_rounds=50,
        use_best_model = True,
    )

    val_pred = model_kf.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_auc_scores.append(fold_auc)
    print(f"Fold {fold+1}: ROC-AUC = {fold_auc:.4f}")

print("Finish:")
print(f"Mean ROC-AUC: {np.mean(fold_auc_scores):.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_train, oof_preds)}")

Fold 1: ROC-AUC = 0.8897
Fold 2: ROC-AUC = 0.8892
Fold 3: ROC-AUC = 0.8908
Fold 4: ROC-AUC = 0.8910
Fold 5: ROC-AUC = 0.8872
Finish:
Mean ROC-AUC: 0.8896
OOF ROC-AUC: 0.8895660376227704


In [5]:
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

encoder = OneHotEncoder()

X_train_encoded = pd.get_dummies(X_train, columns=cat_features, drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test, columns=cat_features, drop_first=True, dtype=int)

oof_preds_xgb = np.zeros(len(X_train))
oof_preds_lgbm = np.zeros(len(X_train))

fold_auc_scores_xgb = []
fold_auc_scores_lgbm = []


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_encoded, y_train)):
    X_tr, X_val = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    xgb = XGBClassifier(
        n_estimators=500,
        random_state=42 
    )

    lgbm = LGBMClassifier(
        n_estimators=500,
        random_state=42,
        verbose=-1
    )

    xgb.fit(X_tr, y_tr)
    lgbm.fit(X_tr, y_tr)

    val_pred_xgb = xgb.predict_proba(X_val)[:, 1]
    val_pred_lgbm = lgbm.predict_proba(X_val)[:, 1]

    oof_preds_xgb[val_idx] = val_pred_xgb
    oof_preds_lgbm[val_idx] = val_pred_lgbm

    fold_auc_xgb = roc_auc_score(y_val, val_pred_xgb)
    fold_auc_lgbm = roc_auc_score(y_val, val_pred_lgbm)

    fold_auc_scores_xgb.append(fold_auc_xgb)
    fold_auc_scores_lgbm.append(fold_auc_lgbm)

    print(f"Fold: {fold+1}")
    print(f"XGBoost: {fold_auc_xgb:.4f}")
    print(f"LGBM: {fold_auc_lgbm:.4f}")
print("XGBoost:")
print(f"Mean ROC-AUC: {np.mean(fold_auc_scores_xgb):.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_train, oof_preds_xgb)}")

print("LGBM:")
print(f"Mean ROC-AUC: {np.mean(fold_auc_scores_lgbm):.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_train, oof_preds_lgbm)}")
    
print("CatBoost")
print(f"Mean ROC-AUC: {np.mean(fold_auc_scores):.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_train, oof_preds)}")

Fold: 1
XGBoost: 0.8747
LGBM: 0.8880
Fold: 2
XGBoost: 0.8746
LGBM: 0.8874
Fold: 3
XGBoost: 0.8766
LGBM: 0.8895
Fold: 4
XGBoost: 0.8759
LGBM: 0.8896
Fold: 5
XGBoost: 0.8723
LGBM: 0.8844
XGBoost:
Mean ROC-AUC: 0.8748
OOF ROC-AUC: 0.8747907474345168
LGBM:
Mean ROC-AUC: 0.8878
OOF ROC-AUC: 0.8877548926670592
CatBoost
Mean ROC-AUC: 0.8896
OOF ROC-AUC: 0.8895660376227704


In [6]:
# feature engenering

# 1. financial features
X_train['Balance_per_Product'] = X_train['Balance'] / (X_train['NumOfProducts']+0.00001)
X_test['Balance_per_Product'] = X_test['Balance'] / (X_test['NumOfProducts']+0.00001)

X_train['Balance_per_Salary'] = X_train['Balance'] / (X_train['EstimatedSalary']+0.00001)
X_test['Balance_per_Salary'] = X_test['Balance'] / (X_test['EstimatedSalary']+0.00001)

X_train['CreditScore_per_Age'] = X_train['CreditScore'] / (X_train['Age']+0.00001)
X_test['CreditScore_per_Age'] = X_test['CreditScore'] / (X_test['Age']+0.00001)

# 2. behavioral features
X_train['IsActiveMember_HasCrCard'] = X_train['IsActiveMember'] * X_train['HasCrCard']
X_test['IsActiveMember_HasCrCard'] = X_test['IsActiveMember'] * X_test['HasCrCard']

X_train['Age_Group'] = X_train['Age'] // 10
X_test['Age_Group'] = X_test['Age'] // 10

X_train.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Balance_per_Product,Balance_per_Salary,CreditScore_per_Age,IsActiveMember_HasCrCard,Age_Group
0,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0.000000,0.000000,20.242418,0.0,3.0
1,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0.000000,0.000000,18.999994,1.0,3.0
2,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0.000000,0.000000,16.949996,0.0,4.0
3,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,148881.051189,1.760655,17.088230,1.0,3.0
4,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0.000000,0.000000,21.696963,1.0,3.0


In [7]:
# Data preparation for LGBM
X_train_freq = X_train.copy()
X_test_freq = X_test.copy()

X_train_freq['Geography_freq'] = X_train_freq['Geography'].map(X_train_freq['Geography'].value_counts())
X_train_freq['Gender_freq'] = X_train_freq['Gender'].map(X_train_freq['Gender'].value_counts())

X_test_freq['Geography_freq'] = X_test_freq['Geography'].map(X_train_freq['Geography'].value_counts())
X_test_freq['Gender_freq'] = X_test_freq['Gender'].map(X_train_freq['Gender'].value_counts())

X_train_freq=X_train_freq.drop(columns=cat_features)
X_test_freq=X_test_freq.drop(columns=cat_features)

X_train_freq.head() 

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Balance_per_Product,Balance_per_Salary,CreditScore_per_Age,IsActiveMember_HasCrCard,Age_Group,Geography_freq,Gender_freq
0,668,33.0,3,0.00,2,1.0,0.0,181449.97,0.000000,0.000000,20.242418,0.0,3.0,94215,93150
1,627,33.0,1,0.00,2,1.0,1.0,49503.50,0.000000,0.000000,18.999994,1.0,3.0,94215,93150
2,678,40.0,10,0.00,2,1.0,0.0,184866.69,0.000000,0.000000,16.949996,0.0,4.0,94215,93150
3,581,34.0,2,148882.54,1,1.0,1.0,84560.88,148881.051189,1.760655,17.088230,1.0,3.0,94215,93150
4,716,33.0,5,0.00,2,1.0,1.0,15068.83,0.000000,0.000000,21.696963,1.0,3.0,36213,93150


In [8]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_cb = np.zeros(len(X_train))
oof_lgbm = np.zeros(len(X_train))

fold_auc_scores_cb = []
fold_auc_scores_lgbm = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

    X_tr_cb, X_val_cb = X_train.iloc[train_idx], X_train.iloc[val_idx]
    X_tr_lgbm, X_val_lgbm = X_train_freq.iloc[train_idx], X_train_freq.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    cb = CatBoostClassifier(
        iterations=500,
        cat_features=cat_features,
        early_stopping_rounds=50,
        random_state=42,
        verbose=0
    )

    lgbm = LGBMClassifier(
        n_estimators=500,
        early_stopping_rounds=50,
        learning_rate=0.04,
        random_state=42,
        verbose=-1
    )

    cb.fit(X_tr_cb, y_tr,
           eval_set=(X_val_cb, y_val)
           )
    lgbm.fit(X_tr_lgbm, y_tr,
             eval_set=[(X_val_lgbm, y_val)]
            )

    val_pred_cb = cb.predict_proba(X_val_cb)[:, 1]
    val_pred_lgbm = lgbm.predict_proba(X_val_lgbm)[:, 1]

    oof_cb[val_idx] = val_pred_cb
    oof_lgbm[val_idx] = val_pred_lgbm

    fold_auc_cb = roc_auc_score(y_val, val_pred_cb)
    fold_auc_lgbm = roc_auc_score(y_val, val_pred_lgbm)

    fold_auc_scores_cb.append(fold_auc_cb)
    fold_auc_scores_lgbm.append(fold_auc_lgbm)

    print(f"Fold: {fold+1}")
    print(f"CatBoost: {fold_auc_cb:.4f}")
    print(f"LightGBM: {fold_auc_lgbm:.4f}")

print('Fit finished:')
print("CatBoost")
print(f"Mean ROC-AUC: {np.mean(fold_auc_scores_cb):.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_train, oof_cb):.4f}")

print("LightGBM")
print(f"Mean ROC-AUC: {np.mean(fold_auc_scores_lgbm):.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_train, oof_lgbm):.4f}")


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold: 1
CatBoost: 0.8896
LightGBM: 0.8899


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold: 2
CatBoost: 0.8894
LightGBM: 0.8894


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold: 3
CatBoost: 0.8905
LightGBM: 0.8910


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold: 4
CatBoost: 0.8911
LightGBM: 0.8907


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold: 5
CatBoost: 0.8872
LightGBM: 0.8867
Fit finished:
CatBoost
Mean ROC-AUC: 0.8895
OOF ROC-AUC: 0.8895
LightGBM
Mean ROC-AUC: 0.8895
OOF ROC-AUC: 0.8895


In [9]:
best_w = 0.5
best_auc = 0.5
for w in np.linspace(0, 1, 101):
    blend_oof = w * oof_cb + (1-w) * oof_lgbm
    score = roc_auc_score(y_train, blend_oof)

    if score > best_auc:
        best_auc = score
        best_w = w

print(f"Оптимальный вес CatBoost: {best_w:.2f}")
print(f"Оптимальный вес LightGBM: {1 - best_w:.2f}")
print(f"OOF ROC-AUC Catboost: {roc_auc_score(y_train, oof_cb):.5f}")
print(f"OOF ROC-AUC LightGBM: {roc_auc_score(y_train, oof_lgbm):.5f}")
print(f"OOF ROC-AUC Blending: {best_auc:.5f}")


Оптимальный вес CatBoost: 0.50
Оптимальный вес LightGBM: 0.50
OOF ROC-AUC Catboost: 0.88953
OOF ROC-AUC LightGBM: 0.88953
OOF ROC-AUC Blending: 0.88993


In [13]:
test_preds_cb = np.zeros(len(X_test))
test_preds_lgbm = np.zeros(len(X_test_freq))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

    X_tr_cb, X_val_cb = X_train.iloc[train_idx], X_train.iloc[val_idx]
    X_tr_lgbm, X_val_lgbm = X_train_freq.iloc[train_idx], X_train_freq.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    cb = CatBoostClassifier(
        iterations=500,
        cat_features=cat_features,
        early_stopping_rounds=50,
        random_state=42,
        verbose=0
    )

    lgbm = LGBMClassifier(
        n_estimators=500,
        early_stopping_rounds=50,
        learning_rate=0.04,
        random_state=42,
        verbose=-1
    )

    cb.fit(X_tr_cb, y_tr,
           eval_set=(X_val_cb, y_val)
           )
    lgbm.fit(X_tr_lgbm, y_tr,
             eval_set=[(X_val_lgbm, y_val)]
            )
    test_preds_cb += cb.predict_proba(X_test)[:, 1] / skf.n_splits
    test_preds_lgbm += lgbm.predict_proba(X_test_freq)[:, 1] / skf.n_splits

final_test_preds = best_w * test_preds_cb + (1-best_w) * test_preds_lgbm

sub = pd.read_csv(r"C:\Users\IVAN\Desktop\Machine-Learning-Roadmap-Theory-and-Practice\Files\Binary Classification with a Bank Churn Dataset\sample_submission.csv")

sub['Exited'] = final_test_preds
sub.to_csv('submission_blend.csv', index=False)

c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated,